In [ ]:
# 課題2
"""
自分の興味のあるデータ（対象は何でも良いが回帰の問題をおすすめする：SHAPバージョンによるトラブルを避けるため）について
今回と同様のRandomForestやSHAP値による解析を行い，対象のデータの予測に効く変数がどのようなものか，
またそこからどのようなことが考察できるのか（どのような現象が発生しているか？など）について考察を行え
"""

In [ ]:
%pip install xgboost
%pip install shap

In [ ]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import shap

In [ ]:
# データ準備
df = pd.read_csv('data/data.csv') 

# ★「予測したい列名」をここで指定してください
target_col = 'Win'

df.describe().T

In [ ]:
# 欠損値の確認
df.info()

In [ ]:
# 不要な変数の削除
df = df.drop(["Year", "League", "Team", "Lose", ], axis=1)
df

In [ ]:
# 説明変数(X)と目的変数(y)に分離
X = df.drop(target_col, axis=1)
y = df[target_col]

# 訓練データとテストデータに分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# SVM用に標準化（回帰でもSVMはスケールの影響を強く受けます）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns) # SHAP用

In [ ]:
# モデル構築（回帰モデルに変更）

# XGBoost Regressor
xgb_model = xgb.XGBRegressor(random_state=42)
xgb_model.fit(X_train, y_train)

# Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42, max_depth=10) # 深さは適宜調整
rf_model.fit(X_train, y_train)

# SVM Regressor (SVR)
# データ数が多いと時間がかかるため、多い場合は訓練データを間引く等の工夫が必要
svm_model = SVR(kernel='rbf') 
svm_model.fit(X_train_scaled, y_train)

In [ ]:
# 評価（R2スコアやRMSEに変更）
models = {'XGBoost': xgb_model, 'Random Forest': rf_model}
preds = {}

# Tree系モデルの評価
for name, model in models.items():
    pred = model.predict(X_test)
    preds[name] = pred
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"[{name}] RMSE: {rmse:.4f}, R2: {r2:.4f}")

# SVMの評価
svm_pred = svm_model.predict(X_test_scaled)
rmse_svm = np.sqrt(mean_squared_error(y_test, svm_pred))
r2_svm = r2_score(y_test, svm_pred)
print(f"[SVM]     RMSE: {rmse_svm:.4f}, R2: {r2_svm:.4f}")

In [ ]:
# 変数重要度の比較
# Gini重要度 (Random Forest & XGBoost)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# XGBoost
pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values().plot(
    kind='barh', ax=axes[0], title='XGBoost Feature Importance'
)

# Random Forest
pd.Series(rf_model.feature_importances_, index=X.columns).sort_values().plot(
    kind='barh', ax=axes[1], title='Random Forest Feature Importance'
)

# SVM (Permutation Importance)
# SVMは係数が見えないため、変数を入れ替えて精度低下を見る手法を使います
perm_importance = permutation_importance(svm_model, X_test_scaled, y_test, n_repeats=5, random_state=42)
pd.Series(perm_importance.importances_mean, index=X.columns).sort_values().plot(
    kind='barh', ax=axes[2], title='SVM Permutation Importance'
)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP値による解析
# XGBoost
explainer_xgb = shap.TreeExplainer(xgb_model)
shap_values_xgb = explainer_xgb.shap_values(X_test)

# Random Forest
explainer_rf = shap.TreeExplainer(rf_model)
shap_values_rf = explainer_rf.shap_values(X_test)

# SVM (KernelExplainer: 計算が重いため、データが多い場合はkmeansで要約するか、sample数を減らしてください)
# ここではテストデータの先頭100件だけで計算する例にします
X_test_sample = X_test_scaled_df.iloc[:100, :] 
# 背景データとして訓練データを要約
X_train_summary = shap.kmeans(X_train_scaled, 10) 
explainer_svm = shap.KernelExplainer(svm_model.predict, X_train_summary)
shap_values_svm = explainer_svm.shap_values(X_test_sample)

In [ ]:

# プロット
print("XGBoost SHAP")
shap.summary_plot(shap_values_xgb, X_test)

print("Random Forest SHAP")
shap.summary_plot(shap_values_rf, X_test)

print("SVM SHAP (Sampled)")
shap.summary_plot(shap_values_svm, X_test_sample)